# [3-009-02] 경도 예측 인공지능 학습 모델

### 실험 기반 인공지능 학습용 데이터 SET을 활용하여 혼합 조성에 따른 금속의 경도 값을 예측하는 인공지능 학습 모델을 제공함

### 1. 라이브러리 불러오기

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import keras
from keras.models import Sequential
from keras.optimizers import Adam, Nadam, SGD, Adamax, Adagrad
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from time import time, ctime
ctime(time())

'Fri Dec 16 00:42:29 2022'

### 2. 학습용 데이터 SET 불러오기

In [2]:
# Importing the dataset
df_h = pd.read_csv(r'Hardness_data_set.csv') # Error 발생할 경우 파일 경로 확인 필수
df_h.head()

,Number,X,Y,Al,Ti,Cr,Fe,Co,Ni,Cu,...,Thickness,Hardness,Modulus,ravg,delta,dHmix,ENavg,dEN,N,Compo
0,1,-35,-25,0.0,26.93146,13.26419,0.0,59.80435,0.0,0.0,...,567.4116,11.475,197.42,0.136581,0.139661,-20.308364,1.780797,0.121591,3,Co/Cr/Ti
1,2,-35,25,0.0,46.79529,12.05821,0.0,41.14650,0.0,0.0,...,504.2531,13.009,208.71,0.145122,0.147847,-23.938973,1.731804,0.124530,3,Co/Cr/Ti
2,3,-35,15,0.0,42.85500,13.00000,0.0,44.14500,0.0,0.0,...,517.3333,13.807,224.48,0.143428,0.148363,-23.666679,1.739977,0.125122,3,Co/Cr/Ti
3,4,-35,-15,0.0,30.14000,13.61500,0.0,56.24500,0.0,0.0,...,552.0000,11.732,231.67,0.137960,0.143021,-21.360750,1.771683,0.123417,3,Co/Cr/Ti
4,5,-35,5,0.0,37.79500,14.24750,0.0,47.95750,0.0,0.0,...,537.7500,14.615,191.88,0.141252,0.147606,-22.901597,1.750388,0.125084,3,Co/Cr/Ti


### 3. 학습용 데이터 전처리

In [3]:
# 학습용 데이터의 크기를 확인한다.
print('학습용 데이터의 크기: ', df_h.shape)

학습용 데이터의 크기:  (1000, 28)


In [4]:
# 학습용 데이터의 특성값을 확인한다.

print('데이터 SET의 전체 특성의 개수: ', df_h.columns.nunique())
print('__________________________________________________________________________')
print(df_h.columns.unique())

데이터 SET의 전체 특성의 개수:  28
__________________________________________________________________________
Index(['Number', 'X', 'Y', 'Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
       'Mo', 'W', 'Mn', 'Si', 'Mg', 'Re', 'Ta', 'Thickness', 'Hardness',
       'Modulus', 'ravg', 'delta', 'dHmix', 'ENavg', 'dEN', 'N', 'Compo'],
      dtype='object')


In [5]:
# 각 컬럼(특성)의 정보를 확인한다.
df_h.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 28 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Number     1000 non-null   int64  
 1   X          1000 non-null   int64  
 2   Y          1000 non-null   int64  
 3   Al         1000 non-null   float64
 4   Ti         1000 non-null   float64
 5   Cr         1000 non-null   float64
 6   Fe         1000 non-null   float64
 7   Co         1000 non-null   float64
 8   Ni         1000 non-null   float64
 9   Cu         1000 non-null   float64
 10  Zr         1000 non-null   float64
 11  Mo         1000 non-null   float64
 12  W          1000 non-null   float64
 13  Mn         1000 non-null   int64  
 14  Si         1000 non-null   int64  
 15  Mg         1000 non-null   float64
 16  Re         1000 non-null   float64
 17  Ta         1000 non-null   float64
 18  Thickness  1000 non-null   float64
 19  Hardness   1000 non-null   float64
 20  Modulus  

In [6]:
# 각 특성별 데이터의 통계값을 확인한다.
df_h.describe().T

,count,mean,std,min,25%,50%,75%,max
Number,1000.0,500.500000,288.819436,1.000000,250.750000,500.500000,750.250000,1000.000000
X,1000.0,-0.157000,19.364848,-45.000000,-15.000000,0.000000,15.000000,35.000000
Y,1000.0,0.300000,18.119556,-43.000000,-13.000000,0.000000,15.000000,43.000000
Al,1000.0,15.294389,25.407708,0.000000,0.000000,0.000000,25.908799,100.000000
Ti,1000.0,10.588386,17.774036,0.000000,0.000000,0.000000,21.692918,70.299380
Cr,1000.0,18.853024,27.026784,0.000000,0.000000,8.619807,27.488383,100.000000
Fe,1000.0,17.504913,30.691831,0.000000,0.000000,0.000000,26.206292,100.000000
Co,1000.0,3.458677,13.057530,0.000000,0.000000,0.000000,0.000000,67.523000
Ni,1000.0,19.370774,29.806494,0.000000,0.000000,0.000000,25.810648,100.000000
Cu,1000.0,1.226543,4.720286,0.000000,0.000000,0.000000,0.000000,35.435178


In [7]:
# 인공지능 학습용 데이터셋에 조성 조합을 확인한다.
df_h['Compo'].unique()

array(['Co/Cr/Ti', 'Ni/Fe/Cr', 'Ni/Mo/W', 'Ni', 'Cu', 'Zr/Cu/Ni/Al',
       'W/Re/Ta', 'W/Re', 'W/Ta', 'Cr', 'Fe', 'Al/Ti/Cr', 'Al', 'W',
       'Al/Ti/Ni', 'Al/Ti/Zr', 'Mg/Al/Zr', 'Mg/Al/Cr', 'Mg/Al/Fe',
       'Mg/Ti/Cr', 'Mg/Al/Ti', 'Mg/Al/Ni', 'Ti/Cr/Fe', 'Ti/Cr/Ni'],
      dtype=object)

In [8]:
# 인공지능 학습을 위하여 각 컬럼별 Null 값을 확인한다.
df_h.isnull().any()

Number       False
X            False
Y            False
Al           False
Ti           False
Cr           False
Fe           False
Co           False
Ni           False
Cu           False
Zr           False
Mo           False
W            False
Mn           False
Si           False
Mg           False
Re           False
Ta           False
Thickness    False
Hardness     False
Modulus      False
ravg         False
delta        False
dHmix        False
ENavg        False
dEN          False
N            False
Compo        False
dtype: bool

### 4. 인공지능 학습용 모델

#### 4-1. 피쳐값 지정

전체 28개 피쳐값 가운데 Number, N, Compo 값은 인공지능 학습에 크게 중요하지 않으므로 사용하지 않는다.
여러 피쳐값을 조합하여 각 조성에 따른 Hardness를 예측한다.

##### Case 1. Number, N, Compo를 제외한 24개의 피쳐값을 사용하여 Hardness 예측
##### Case 2. Thickness, Modulus, ravg, delta, dHmix, ENavg, dEN 사용하여 Hardness 예측
##### Case 3. Modulus, delta, dHmix, ENavg, dEN 사용하여 Hardness 예측
##### Case 4. Modulus, delta, ENavg 사용하여 Hardness 예측
#####  Case 5. delta 사용하여 Hardness 예측

** 참고사항, ρ(로우)는 물리학에서 비저항(Resistivity)을 나타낸다. **

In [9]:
# 사용 안하는 3가지 피쳐값을 제외한다.
df = df_h.drop(['Number', 'N', 'Compo'], axis = 1)
df.head()

,X,Y,Al,Ti,Cr,Fe,Co,Ni,Cu,Zr,...,Re,Ta,Thickness,Hardness,Modulus,ravg,delta,dHmix,ENavg,dEN
0,-35,-25,0.0,26.93146,13.26419,0.0,59.80435,0.0,0.0,0.0,...,0.0,0.0,567.4116,11.475,197.42,0.136581,0.139661,-20.308364,1.780797,0.121591
1,-35,25,0.0,46.79529,12.05821,0.0,41.14650,0.0,0.0,0.0,...,0.0,0.0,504.2531,13.009,208.71,0.145122,0.147847,-23.938973,1.731804,0.124530
2,-35,15,0.0,42.85500,13.00000,0.0,44.14500,0.0,0.0,0.0,...,0.0,0.0,517.3333,13.807,224.48,0.143428,0.148363,-23.666679,1.739977,0.125122
3,-35,-15,0.0,30.14000,13.61500,0.0,56.24500,0.0,0.0,0.0,...,0.0,0.0,552.0000,11.732,231.67,0.137960,0.143021,-21.360750,1.771683,0.123417
4,-35,5,0.0,37.79500,14.24750,0.0,47.95750,0.0,0.0,0.0,...,0.0,0.0,537.7500,14.615,191.88,0.141252,0.147606,-22.901597,1.750388,0.125084


In [10]:
# AI 성능 평가를 위한 함수 

def score_matrix(y_real, y_pred, X_test):
    print("MAE: ", mean_absolute_error(y_real, y_pred))
    SSE = np.sum((y_real - y_pred)**2)
    SSR = np.sum((y_pred - np.mean(y_real))**2)
    print("R2 Score :", 1-SSE/SSR)
    pred_rsq = 1 - np.sum(np.square(y_real-y_pred)) / np.var(y_real) / y_real.size
    print("Predicted R2 Score :", pred_rsq)
    p = X_test.shape[1]
    adj_r2 = 1-(SSE/SSR) * (len(y_real)-1) / (len(y_real) - p - 1)
    print("adjust R2 Score :", adj_r2)
    
data_set = []

In [11]:
# Case 1. Number, N, Compo를 제외한 22개의 피쳐값을 사용하여 Hardness 예측
X1 = np.array(df[[ 'X', 'Y', 'Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
       'Mo', 'W', 'Mn', 'Si', 'Mg', 'Modulus', 'Thickness',
       'ravg', 'delta', 'dHmix', 'ENavg', 'dEN']])
X1 = X1.astype(float)
y1 = np.array(df["Hardness"])
y1 = y1.astype(float)
data_set.append(train_test_split(X1, y1, test_size = 0.4, random_state=42))

In [12]:
# Case 2. 'Modulus', Thickness, ravg, delta, dHmix, ENavg, dEN 사용하여 Hardness 예측
X2 = np.array(df[['Modulus', "Thickness", "ravg", "delta", "dHmix", "ENavg", "dEN"]]) # 7 Features
X2 = X2.astype(float)
y2 = np.array(df["Hardness"])
y2 = y2.astype(float)
data_set.append(train_test_split(X2, y2, test_size = 0.2, random_state=42))

In [13]:
# Case 3. 'Modulus',"delta", "dHmix", "ENavg", "dEN" 5가지 특성값을 사용하여 Hardness 예측
X3 = np.array(df[['Modulus', "delta", "dHmix", "ENavg", "dEN"]]) # 5Features
X3 = X3.astype(float)
y3 = np.array(df["Hardness"])
y3 = y3.astype(float)
data_set.append(train_test_split(X3, y3, test_size = 0.2, random_state=38))

In [14]:
# Case 4. 'Modulus', delta, ENavg 사용하여 Hardness 예측
X4 = np.array(df[['Modulus', "delta", "ENavg"]]) # 3 Features
X4 = X4.astype(float)
y4 = np.array(df["Hardness"])
y4 = y4.astype(float)
data_set.append(train_test_split(X4, y4, test_size = 0.2, random_state=38))

In [15]:
# Case 5. delta 사용하여 Hardness 예측
X5 = np.array(df[["delta"]]) # 1 Features
X5 = X5.astype(float)
y5 = np.array(df["Hardness"])
y5 = y5.astype(float)
data_set.append(train_test_split(X5, y5, test_size = 0.2, random_state=38))

In [16]:
# Model 1. LinearRegression 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model = LinearRegression()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  0.9955635021854691
R2 Score : 0.6146584488876743
Predicted R2 Score : 0.715035735595825
adjust R2 Score : 0.5921716740217031
Features Case_ 2
MAE:  1.7671161526870054
R2 Score : -2.0426330207384287
Predicted R2 Score : 0.2207220612383981
adjust R2 Score : -2.153562349619517
Features Case_ 3
MAE:  1.4994042774366694
R2 Score : -1.9871541316629524
Predicted R2 Score : 0.3155728578473169
adjust R2 Score : -2.0641426402109664
Features Case_ 4
MAE:  1.826478586204333
R2 Score : -4.473427774081368
Predicted R2 Score : 0.09611082991468867
adjust R2 Score : -4.557204729807103
Features Case_ 5
MAE:  1.9385907478219593
R2 Score : -7.788828326810757
Predicted R2 Score : 0.0030633393608177073
adjust R2 Score : -7.833216348663337


In [17]:
# Model 2. DecisionTreeRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model = DecisionTreeRegressor()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  0.85270025
R2 Score : 0.6383641919444272
Predicted R2 Score : 0.6253127784395819
adjust R2 Score : 0.6172607760897253
Features Case_ 2
MAE:  0.9863595000000001
R2 Score : 0.5469004829201831
Predicted R2 Score : 0.5831776250916063
adjust R2 Score : 0.5303812296933148
Features Case_ 3
MAE:  0.8608754999999999
R2 Score : 0.6887777019265191
Predicted R2 Score : 0.6704931473754131
adjust R2 Score : 0.6807565086772027
Features Case_ 4
MAE:  0.9627439999999998
R2 Score : 0.6595815298808598
Predicted R2 Score : 0.6374769634428983
adjust R2 Score : 0.654371043093322
Features Case_ 5
MAE:  1.676711936619718
R2 Score : -0.85955567622622
Predicted R2 Score : 0.11630261136602615
adjust R2 Score : -0.8689473715606959


In [18]:
# Model 3. RandomForestRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  RandomForestRegressor(n_estimators=25, n_jobs=-1, verbose=1)
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  0.6835173999999999
R2 Score : 0.7414180918181874
Predicted R2 Score : 0.7716925803159502
adjust R2 Score : 0.7263284313937846
Features Case_ 2
MAE:  0.7817622199999996
R2 Score : 0.6691098012774674
Predicted R2 Score : 0.7543531154740127
adjust R2 Score : 0.6570460961157085
Features Case_ 3


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished


MAE:  0.7278394200000001
R2 Score : 0.7749703904930453
Predicted R2 Score : 0.7936579874514915
adjust R2 Score : 0.7691706582892578
Features Case_ 4
MAE:  0.7889330600000001
R2 Score : 0.7720660398486376
Predicted R2 Score : 0.791876939875018
adjust R2 Score : 0.7685772547442801
Features Case_ 5
MAE:  1.589792440874835
R2 Score : -0.9119486041421339
Predicted R2 Score : 0.1776731215419235
adjust R2 Score : -0.9216049102236599


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 out of  25 | elapsed:    0.0s finished


In [25]:
# Model 4. Lasso 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Lasso()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  1.1652641117474887
R2 Score : 0.35702892265607966
Predicted R2 Score : 0.6381293726514545
adjust R2 Score : 0.31950806403123544
Features Case_ 2
MAE:  2.1725879923811577
R2 Score : -10.452166983453596
Predicted R2 Score : -0.037651371294948666
adjust R2 Score : -10.869693904725343
Features Case_ 3
MAE:  1.9033590187932308
R2 Score : -13.100663143282404
Predicted R2 Score : 0.05326315037197771
adjust R2 Score : -13.464082296459786
Features Case_ 4
MAE:  1.951364826743473
R2 Score : -15.360461182004443
Predicted R2 Score : 0.0133719888759406
adjust R2 Score : -15.61087640417798
Features Case_ 5
MAE:  2.03353010375
R2 Score : -147.45478013174608
Predicted R2 Score : -0.006781740131493352
adjust R2 Score : -148.2045517485731


In [20]:
# Model 5. Ridge 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Ridge(alpha = 0.1)
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

Features Case_ 1
MAE:  0.9989473245168196
R2 Score : 0.6051736973630496
Predicted R2 Score : 0.7138908772859567
adjust R2 Score : 0.5821334356707077
Features Case_ 2
MAE:  1.7707419580815524
R2 Score : -2.1148911428274184
Predicted R2 Score : 0.21866237037098268
adjust R2 Score : -2.228454882409668
Features Case_ 3
MAE:  1.5071205417924043
R2 Score : -2.0731809551583873
Predicted R2 Score : 0.31235391880382923
adjust R2 Score : -2.1523866498789643
Features Case_ 4
MAE:  1.8262628883396657
R2 Score : -4.493977891479434
Predicted R2 Score : 0.09592198123444262
adjust R2 Score : -4.578069389818405
Features Case_ 5
MAE:  1.9384752748432228
R2 Score : -7.820187733315892
Predicted R2 Score : 0.0032557320882586893
adjust R2 Score : -7.864734136009407


In [21]:
ctime(time())

'Fri Dec 16 00:42:32 2022'